# Multi-Event Ridge Regression — Storm-Surge Error Model

Pipeline **event-focused** per modellare *solo* l'errore dinamico del modello idrodinamico
(HDM) durante le mareggiate, su una singola stazione mareografica.

### Obiettivo scientifico
Stimare un modello lineare comune a **tutti** gli eventi di mareggiata della finestra di
training (~2 anni di dati orari), così che le funzioni di risposta (IRF) catturate siano
**proprietà del sistema** e non di un singolo evento. Il modello viene poi testato su
eventi **mai visti** (cross-event out-of-sample).

### Target
$$\varepsilon(t) \;=\; \text{HDM}(t) - \text{TG}(t) \;=\; \text{forcoast\_p82\_m}(t) - \text{tg\_obs\_m}(t)$$

### Modello (MISO lineare con lag finiti, regolarizzato Ridge)
$$
\varepsilon(t) \;=\; \beta_0 \;+\; \sum_{i=1}^{n_f}\sum_{k=0}^{L} \beta_{i,k}\, x_i(t-k) \;+\; \eta(t),
\qquad n_f=4,\ L=72\ \text{h} \;\Rightarrow\; p = n_f(L+1) = 292
$$
dove $x_i \in \{\text{SLP},\ t_{2m},\ u_{10},\ v_{10}\}$.

### Stima
$$
\hat{\boldsymbol\beta} \;=\; \arg\min_{\boldsymbol\beta}\Big\{\|\mathbf{y} - Z\boldsymbol\beta\|_2^2 + \alpha\|\boldsymbol\beta\|_2^2\Big\}
\;=\; (Z^\top Z + \alpha I_p)^{-1} Z^\top \mathbf{y}
$$
con $Z$ matrice standardizzata e $\alpha$ scelto via **GroupKFold** (le righe dello stesso
evento finiscono sempre insieme — niente leakage).

### Vincoli metodologici
- Solo finestre temporali attorno ai picchi (no addestramento sulla serie continua di 2 anni).
- Lag costruiti **dentro** ogni finestra (mai a cavallo di due eventi).
- Un solo modello Ridge condiviso fra tutti gli eventi di training.
- CV **per gruppo** (event ID), no random shuffle.
- Test su eventi disgiunti, stessa ricetta di lagging, **senza re-fit**.
- Niente deep learning, niente satellite, niente correzione statica del bias.


In [1]:
"""Setup, imports and station loading."""
from pathlib import Path
from dataclasses import dataclass, field
from typing import Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from statsmodels.stats.stattools import durbin_watson
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})

# ── Station ────────────────────────────────────────────────────────────
DATA_DIR     = Path("/Users/nicolocaron/Desktop/MASTER PROJECT/data/per_station")
STATION_FILE = "station_30336_Kobenhavn.parquet"   # <-- change to use another station

# Load
df = pd.read_parquet(DATA_DIR / STATION_FILE)
df["time"]    = pd.to_datetime(df["time"]) 
df            = df.sort_values("time").reset_index(drop=True)
df["error_m"] = df["forcoast_p82_m"] - df["tg_obs_m"]   # ε(t) = HDM − TG

# ── Sanity filter: alcune stazioni hanno valori sentinel; set NaN se fuori range
PHYS_RANGES = {
    "SLP": (90_000.0, 110_000.0),    # Pa
    "t2m": (220.0,    320.0),        # K
    "u10": (-50.0,    50.0),         # m/s
    "v10": (-50.0,    50.0),         # m/s
}
n_before = len(df)
mask_bad = pd.Series(False, index=df.index)
for col, (lo, hi) in PHYS_RANGES.items():
    if col in df.columns:
        bad = (df[col] < lo) | (df[col] > hi)
        if bad.any():
            print(f"  ⚠ {col}: {bad.sum()} righe fuori range [{lo}, {hi}] → set NaN")
            df.loc[bad, col] = np.nan
            mask_bad |= bad
print(f"Righe totali con almeno un outlier fisico: {mask_bad.sum()} / {n_before}")

# Sanity check: SLP è la pressione atmosferica (Pa) e DEVE esserci.
if "SLP" not in df.columns:
    raise RuntimeError("Missing 'SLP' column (sea level pressure, Pa)")

# Wind-stress proxy (∝ |U| · U_component)
wind_speed   = np.sqrt(df["u10"].to_numpy()**2 + df["v10"].to_numpy()**2)
df["wstr_u"] = df["u10"].to_numpy() * wind_speed
df["wstr_v"] = df["v10"].to_numpy() * wind_speed

# ── DTU10 ocean tide prediction (replaces naive 12/24h sinusoids) ─────────
# Model files: /Volumes/DTU10_TIDEMODEL/OCEAN_TIDE_GRIDS/GOT47_FORMAT/{M2,S2,K1,O1}.d
# Method: bilinear interp of amp/phase grid at station (lat,lon) →
# astronomical args + nodal corrections (GOT) via pyTMD → tide(t) [m].
import pyTMD.constituents
from scipy.interpolate import RegularGridInterpolator

DTU10_DIR    = "/Volumes/DTU10_TIDEMODEL/OCEAN_TIDE_GRIDS/GOT47_FORMAT"
DTU10_CONST  = ["m2", "s2", "k1", "o1"]                  # 4 dominant components
DTU10_FILES  = ["M2.d", "S2.d", "K1.d", "O1.d"]
# tidal_12h ← M2+S2 component, tidal_24h ← K1+O1 component
DTU10_GROUPS = {"tide_12h": ["m2", "s2"], "tide_24h": ["k1", "o1"]}

def _read_dtu10_got_grid(filepath: str):
    """Read DTU10 GOT4.7 (.d) file → amp [cm], phase [deg], lat, lon vectors."""
    with open(filepath, "r") as f:
        lines = f.readlines()
    ny, nx = int(lines[2].split()[0]), int(lines[2].split()[1])
    lat_min, lat_max = float(lines[3].split()[0]), float(lines[3].split()[1])
    lon_min, lon_max = float(lines[4].split()[0]), float(lines[4].split()[1])
    undef = float(lines[5].strip())
    lat_vec = np.linspace(lat_min, lat_max, ny)
    lon_vec = np.linspace(lon_min, lon_max, nx)
    amp = np.zeros((ny, nx)); phase = np.zeros((ny, nx))
    for i in range(ny):
        amp[i, :] = np.fromstring(lines[7 + i], sep=" ")
    phase_start = 7 + ny + 7
    for i in range(ny):
        phase[i, :] = np.fromstring(lines[phase_start + i], sep=" ")
    amp[amp     >= undef - 0.01] = np.nan
    phase[phase >= undef - 0.01] = np.nan
    return amp, phase, lat_vec, lon_vec

def _dtu10_per_constituent(times_utc, lat, lon, grid_dir, const_names, filenames):
    """Return dict {const_name: tide_series_m} (no sum), so we can group later."""
    pt = np.array([[lat, lon]])
    amps_m, phases_rad = [], []
    for fname in filenames:
        amp_g, ph_g, lat_v, lon_v = _read_dtu10_got_grid(f"{grid_dir}/{fname}")
        ia = RegularGridInterpolator((lat_v, lon_v), amp_g,
                                     method="linear", bounds_error=False, fill_value=np.nan)
        ip = RegularGridInterpolator((lat_v, lon_v), ph_g,
                                     method="linear", bounds_error=False, fill_value=np.nan)
        amps_m.append(ia(pt)[0] / 100.0)                 # cm → m
        phases_rad.append(np.deg2rad(ip(pt)[0]))
    amps_m     = np.array(amps_m)
    phases_rad = np.array(phases_rad)

    t0_ref = pd.Timestamp("1992-01-01")
    t_days = (pd.to_datetime(times_utc) - t0_ref).dt.total_seconds().values / 86400.0
    MJD    = t_days + 48622.0
    pu, pf, G = pyTMD.constituents.arguments(MJD, const_names, corrections="GOT")
    theta  = np.radians(G) + pu                          # (n_times, n_const)
    return {
        const_names[i]: pf[:, i] * amps_m[i] * np.cos(theta[:, i] - phases_rad[i])
        for i in range(len(const_names))
    }

# Coordinate stazione
LAT_ST = float(df["lat"].iloc[0])  if "lat" in df.columns else float(df["latitude"].iloc[0])
LON_ST = float(df["lon"].iloc[0])  if "lon" in df.columns else float(df["longitude"].iloc[0])
print(f"Calcolo marea DTU10 ({', '.join(c.upper() for c in DTU10_CONST)})  "
      f"@ lat={LAT_ST:.3f}, lon={LON_ST:.3f}  ...", end=" ", flush=True)
_per_const = _dtu10_per_constituent(df["time"], LAT_ST, LON_ST,
                                    DTU10_DIR, DTU10_CONST, DTU10_FILES)
df["tide_dtu10_m"] = sum(_per_const.values())            # marea totale (4 cost.)
df["tidal_12h"]    = sum(_per_const[c] for c in DTU10_GROUPS["tide_12h"])  # M2+S2
df["tidal_24h"]    = sum(_per_const[c] for c in DTU10_GROUPS["tide_24h"])  # K1+O1
print(f"OK  (std totale = {df['tide_dtu10_m'].std()*100:.2f} cm; "
      f"12h std = {df['tidal_12h'].std()*100:.2f} cm; "
      f"24h std = {df['tidal_24h'].std()*100:.2f} cm)")

STATION_NAME = df["station_name"].iloc[0]
print(f"Stazione           : {STATION_NAME}")
print(f"Periodo disponibile: {df['time'].min()}  →  {df['time'].max()}")
print(f"N ore totali       : {len(df):,}")

# Top peaks
top = df.nlargest(15, "tg_obs_m")[["time", "tg_obs_m", "forcoast_p82_m", "error_m"]]
print("\nTop 15 picchi TG osservati (per scegliere train/test peaks):")
print(top.to_string(index=False))

  ⚠ SLP: 18 righe fuori range [90000.0, 110000.0] → set NaN
  ⚠ t2m: 18 righe fuori range [220.0, 320.0] → set NaN
Righe totali con almeno un outlier fisico: 18 / 63335
Calcolo marea DTU10 (M2, S2, K1, O1)  @ lat=55.704, lon=12.599  ... OK  (std totale = 2.13 cm; 12h std = 1.69 cm; 24h std = 1.29 cm)
Stazione           : København
Periodo disponibile: 2013-01-01 01:00:00  →  2020-03-31 23:00:00
N ore totali       : 63,335

Top 15 picchi TG osservati (per scegliere train/test peaks):
               time  tg_obs_m  forcoast_p82_m   error_m
2013-12-06 19:00:00      1.66        1.482351 -0.177649
2013-12-06 17:00:00      1.65        1.408141 -0.241859
2013-12-06 16:00:00      1.60        1.368155 -0.231845
2013-12-06 18:00:00      1.60        1.453388 -0.146612
2013-12-06 09:00:00      1.59        1.465829 -0.124171
2013-12-06 10:00:00      1.59        1.444099 -0.145901
2013-12-06 20:00:00      1.57        1.475796 -0.094204
2013-12-06 15:00:00      1.56        1.350795 -0.209205
2013-12-

## 1. Configurazione e funzioni di pipeline

Tutto è configurabile dal `Config` qui sotto. Le funzioni sono pure (no globals)
così il pipeline può essere riusato/testato facilmente.


In [2]:
"""Configurazione & funzioni della pipeline multi-evento (con supporto ARX, HDM-as-feature)."""

from scipy.signal import find_peaks

# ── 1.1 Config ─────────────────────────────────────────────────────────
@dataclass
class Config:
    # Forzanti atmosferiche di base
    features:         tuple = ("SLP", "t2m", "u10", "v10")
    target:           str   = "error_m"                     # ε(t) = HDM − TG
    # Lag NON consecutivi: cattura dinamica veloce + lenta con pochi parametri
    lags:             tuple = (0, 1, 2, 3, 6, 12, 24, 48, 72, 96, 120, 144, 168)
    # Wind-stress derivate (∝ |U|·U_component)
    add_wind_stress:  bool  = True
    # Aggiunge HDM come feature esogena (forcoast_p82_m): il modello impara a
    # correggere il proprio bias DINAMICO usando lo stato corrente dell'HDM
    include_hdm:      bool  = False
    # Lag autoregressivi del TARGET ε (modello ARX). Lista vuota = niente AR.
    # Solo lag ≥ 1 (lag 0 sarebbe trivialmente y(t) sul lato destro).
    ar_lags:          tuple = ()
    pre_event_hours:  int   = 240
    post_event_hours: int   = 240
    cv_n_splits:      int   = 3                              # GroupKFold splits (3 = robusto con ~15 eventi)
    alphas:           tuple = tuple(np.logspace(-3, 3, 25))
    peak_height_m:        float = 0.8
    peak_min_distance_h:  int   = 72
    weight_mode:          str   = "equal_event"

    @property
    def L(self) -> int:
        """Lag massimo (tra exo e AR) per il drop iniziale delle righe NaN."""
        m = max(self.lags)
        if self.ar_lags:
            m = max(m, max(self.ar_lags))
        return m

    @property
    def all_features(self) -> list[str]:
        """Lista completa delle feature esogene (no AR)."""
        feats = list(self.features)
        if self.add_wind_stress:
            feats += ["wstr_u", "wstr_v"]
        if self.include_hdm:
            feats += ["forcoast_p82_m"]
        return feats

    @property
    def n_coef(self) -> int:
        return len(self.all_features) * len(self.lags) + len(self.ar_lags)

CFG = Config()
print(f"Features esogene : {CFG.all_features}")
print(f"Lag set (exo)    : {CFG.lags}  ({len(CFG.lags)} lag/feat)")
print(f"AR lags          : {CFG.ar_lags}")
print(f"p (n. coef)      : {CFG.n_coef}")
print(f"L_max (drop)     : {CFG.L} h")
print(f"Window           : −{CFG.pre_event_hours} / +{CFG.post_event_hours} h")
print(f"CV splits        : {CFG.cv_n_splits} (GroupKFold by event)")
print(f"Peak threshold   : tg_obs_m > {CFG.peak_height_m} m, min dist {CFG.peak_min_distance_h} h")
print(f"Sample weights   : {CFG.weight_mode}")


# ── 1.2 Peak detection ────────────────────────────────────────────────
def detect_peaks(df: pd.DataFrame, cfg: Config,
                 time_from: pd.Timestamp, time_to: pd.Timestamp) -> pd.DataFrame:
    sub = df[(df["time"] >= time_from) & (df["time"] < time_to)].reset_index(drop=True)
    y   = sub["tg_obs_m"].to_numpy()
    idx, props = find_peaks(y, height=cfg.peak_height_m, distance=cfg.peak_min_distance_h)
    return pd.DataFrame({
        "peak"    : sub["time"].iloc[idx].values,
        "height_m": props["peak_heights"],
    }).sort_values("peak").reset_index(drop=True)


# ── 1.3 Estrazione finestre evento ────────────────────────────────────
def extract_event_windows(df: pd.DataFrame, peaks: list, cfg: Config,
                          clip_overlap: bool = True) -> list[dict]:
    peaks = sorted(pd.to_datetime(peaks))
    extras = [c for c in ("forcoast_p82_m", "tg_obs_m") if c in df.columns
              and c not in cfg.all_features]
    needed_cols = ["time", cfg.target] + cfg.all_features + extras
    events = []
    for i, pk in enumerate(peaks):
        start = pk - pd.Timedelta(hours=cfg.pre_event_hours)
        end   = pk + pd.Timedelta(hours=cfg.post_event_hours)
        if clip_overlap:
            if i > 0:
                mid_prev = peaks[i-1] + (pk - peaks[i-1]) / 2
                start    = max(start, mid_prev)
            if i < len(peaks) - 1:
                mid_next = pk + (peaks[i+1] - pk) / 2
                end      = min(end, mid_next)
        sub = df[(df["time"] >= start) & (df["time"] < end)][needed_cols].copy()
        sub = sub.dropna().reset_index(drop=True)
        if len(sub) < cfg.L + 24:
            continue
        events.append({"event_id": i, "peak": pk, "start": start, "end": end, "df": sub})
    return events


# ── 1.4 Lag matrix per evento (exo + AR del target) ───────────────────
def build_lag_matrix(sub: pd.DataFrame, features: list[str], lags: tuple,
                     target: str, ar_lags: tuple = ()):
    """Costruisce X (n−L, p) con lag delle feature esogene + lag del target (ARX)."""
    Lexo = max(lags) if len(lags) else 0
    Lar  = max(ar_lags) if len(ar_lags) else 0
    L    = max(Lexo, Lar)
    cols, names = [], []
    for f in features:
        v = sub[f].to_numpy()
        for k in lags:
            shifted = np.empty_like(v, dtype=float); shifted[:k] = np.nan
            if k > 0: shifted[k:] = v[:-k]
            else:     shifted[:]  = v
            cols.append(shifted); names.append(f"{f}_lag{k}")
    # AR lag del target ε (lag ≥ 1 sempre)
    if len(ar_lags):
        y_full = sub[target].to_numpy()
        for k in ar_lags:
            shifted = np.empty_like(y_full, dtype=float); shifted[:k] = np.nan
            shifted[k:] = y_full[:-k]
            cols.append(shifted); names.append(f"{target}_arlag{k}")
    X = np.column_stack(cols)[L:]
    y = sub[target].to_numpy()[L:]
    t = sub["time"].to_numpy()[L:]
    return X, y, t, names


# ── 1.5 Stack di tutti gli eventi ─────────────────────────────────────
def stack_event_matrices(events: list[dict], cfg: Config) -> dict:
    Xs, ys, ts, gs = [], [], [], []
    per_event, cursor, feat_names = {}, 0, None
    for ev in events:
        Xe, ye, te, names = build_lag_matrix(ev["df"], cfg.all_features, cfg.lags,
                                             cfg.target, cfg.ar_lags)
        if Xe.shape[0] == 0:
            per_event[ev["event_id"]] = (cursor, cursor); continue
        Xs.append(Xe); ys.append(ye); ts.append(te)
        gs.append(np.full(Xe.shape[0], ev["event_id"], dtype=int))
        per_event[ev["event_id"]] = (cursor, cursor + Xe.shape[0])
        cursor += Xe.shape[0]; feat_names = names
    X = np.vstack(Xs); y = np.concatenate(ys); t = np.concatenate(ts); g = np.concatenate(gs)
    return {"X": X, "y": y, "t": t, "groups": g,
            "per_event": per_event, "feat_names": feat_names}


# ── 1.6 Sample weights ────────────────────────────────────────────────
def compute_sample_weights(stack: dict, cfg: Config,
                           peak_heights: dict | None = None) -> np.ndarray:
    n = stack["y"].shape[0]; g = stack["groups"]; w = np.ones(n)
    if cfg.weight_mode == "none":
        return w
    counts = pd.Series(g).value_counts().to_dict()
    if cfg.weight_mode == "equal_event":
        for i in range(n): w[i] = 1.0 / counts[g[i]]
    elif cfg.weight_mode == "intensity":
        assert peak_heights is not None
        for i in range(n): w[i] = peak_heights.get(int(g[i]), 1.0) / counts[g[i]]
    else:
        raise ValueError(f"weight_mode sconosciuto: {cfg.weight_mode}")
    return w * (n / w.sum())


# ── 1.7 Fit Ridge con GroupKFold-CV manuale ───────────────────────────
def fit_ridge_multi_event(stack: dict, cfg: Config,
                          sample_weight: np.ndarray) -> dict:
    X, y, g = stack["X"], stack["y"], stack["groups"]
    n_groups = len(np.unique(g))
    n_splits = min(cfg.cv_n_splits, n_groups)
    gkf = GroupKFold(n_splits=n_splits)
    scores = np.zeros(len(cfg.alphas))
    for ai, alpha in enumerate(cfg.alphas):
        fold = []
        for tr, va in gkf.split(X, y, groups=g):
            scl = StandardScaler().fit(X[tr])
            mdl = Ridge(alpha=alpha).fit(scl.transform(X[tr]), y[tr],
                                         sample_weight=sample_weight[tr])
            yhat = mdl.predict(scl.transform(X[va]))
            fold.append(r2_score(y[va], yhat, sample_weight=sample_weight[va]))
        scores[ai] = np.mean(fold)
    best = int(np.argmax(scores)); alpha_star = cfg.alphas[best]
    scaler = StandardScaler().fit(X)
    model  = Ridge(alpha=alpha_star).fit(scaler.transform(X), y, sample_weight=sample_weight)
    return {"model": model, "scaler": scaler, "alpha": alpha_star,
            "cv_score_best": float(scores[best]),
            "cv_scores": dict(zip(cfg.alphas, scores)),
            "coef": model.coef_, "intercept": model.intercept_,
            "feat_names": stack["feat_names"], "n_splits": n_splits,
            "weight_mode": cfg.weight_mode, "lags": cfg.lags,
            "ar_lags": cfg.ar_lags, "features": cfg.all_features}


# ── 1.8 Valutazione (globale + per evento) ────────────────────────────
def evaluate_model(stack: dict, trained: dict):
    X, y = stack["X"], stack["y"]
    y_pred = trained["model"].predict(trained["scaler"].transform(X))
    res    = y - y_pred
    rows = []
    for ev_id, (a, b) in stack["per_event"].items():
        if b <= a: continue
        ye, yh = y[a:b], y_pred[a:b]
        rows.append({"event_id": ev_id, "n": b-a,
                     "R2"  : r2_score(ye, yh),
                     "RMSE": float(np.sqrt(np.mean((ye-yh)**2))),
                     "MAE" : float(np.mean(np.abs(ye-yh))),
                     "bias": float(np.mean(yh - ye)),
                     "DW"  : float(durbin_watson(ye - yh))})
    glob = {"R2": r2_score(y, y_pred),
            "RMSE": float(np.sqrt(np.mean(res**2))),
            "MAE" : float(np.mean(np.abs(res))),
            "bias": float(np.mean(res)),
            "DW"  : float(durbin_watson(res)),
            "n"   : len(y)}
    return pd.DataFrame(rows), glob, y_pred, res


# ── 1.9 IRF ───────────────────────────────────────────────────────────
def plot_irfs(trained: dict, title_suffix: str = "") -> dict:
    feats = trained["features"]; lags = list(trained["lags"]); coef = trained["coef"]
    p_per = len(lags)
    beta = {f: coef[i*p_per:(i+1)*p_per] for i, f in enumerate(feats)}
    n_f = len(feats); ncols = 3; nrows = int(np.ceil(n_f / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2*ncols, 3.0*nrows), squeeze=False)
    for i, f in enumerate(feats):
        ax = axes[i//ncols, i%ncols]
        ax.plot(lags, beta[f], marker="o", lw=1.2, color="#2C3E50")
        ax.axhline(0, color="grey", lw=0.6)
        ax.set_title(f"IRF — {f}", fontweight="bold")
        ax.set_xlabel("lag [h]"); ax.set_ylabel("β̂  [m / σ]")
        ax.grid(alpha=0.3)
    for j in range(n_f, nrows*ncols):
        axes[j//ncols, j%ncols].axis("off")
    fig.suptitle(f"Impulse Response Functions {title_suffix}", fontweight="bold", y=1.02)
    plt.tight_layout(); plt.show()
    return beta


# ── 1.10 Diagnostica residui ──────────────────────────────────────────
def plot_residual_diagnostics(res: np.ndarray, t: np.ndarray, title: str = "") -> None:
    fig, axes = plt.subplots(2, 2, figsize=(13, 7))
    axes[0,0].plot(t, res, color="#34495E", lw=0.5); axes[0,0].axhline(0, color="red", lw=0.6)
    axes[0,0].set_title("Residui vs tempo"); axes[0,0].set_ylabel("ε − ε̂  [m]")
    axes[0,1].hist(res, bins=60, color="#5DADE2", edgecolor="white")
    axes[0,1].axvline(0, color="red", lw=0.8)
    axes[0,1].set_title(f"Istogramma  (μ={res.mean():+.4f}, σ={res.std():.4f})")
    plot_acf (res, lags=72, ax=axes[1,0]); axes[1,0].set_title("ACF residui")
    plot_pacf(res, lags=72, ax=axes[1,1], method="ywm"); axes[1,1].set_title("PACF residui")
    fig.suptitle(title, fontweight="bold", y=1.00); plt.tight_layout(); plt.show()


# ── 1.11 Esperimento comparativo: griglia di configurazioni ───────────
def run_one_experiment(df: pd.DataFrame, base_cfg_kwargs: dict,
                       train_years: int, label: str) -> dict:
    """Una run completa (train + test out-of-sample) → riga di metriche."""
    cfg = Config(**base_cfg_kwargs)
    t_min = df["time"].min(); t_split = t_min + pd.DateOffset(years=train_years)
    t_max = df["time"].max()
    pk_tr = detect_peaks(df, cfg, t_min, t_split)
    pk_te = detect_peaks(df, cfg, t_split, t_max + pd.Timedelta(hours=1))
    ev_tr = extract_event_windows(df, pk_tr["peak"].tolist(), cfg, clip_overlap=True)
    ev_te = extract_event_windows(df, pk_te["peak"].tolist(), cfg, clip_overlap=True)
    if len(ev_tr) < cfg.cv_n_splits or len(ev_te) == 0:
        return {"label": label, "status": "skipped (eventi insuff.)",
                "n_train_ev": len(ev_tr), "n_test_ev": len(ev_te)}
    st_tr = stack_event_matrices(ev_tr, cfg)
    st_te = stack_event_matrices(ev_te, cfg)
    ph    = {e["event_id"]: float(pk_tr.iloc[e["event_id"]]["height_m"]) for e in ev_tr}
    sw_tr = compute_sample_weights(st_tr, cfg, peak_heights=ph)
    trained = fit_ridge_multi_event(st_tr, cfg, sample_weight=sw_tr)
    _, gtr, _, _ = evaluate_model(st_tr, trained)
    _, gte, _, _ = evaluate_model(st_te, trained)
    return {
        "label"     : label,
        "n_train_ev": len(ev_tr), "n_test_ev": len(ev_te),
        "p_coef"    : st_tr["X"].shape[1], "N_train": st_tr["X"].shape[0],
        "p_over_N"  : st_tr["X"].shape[1] / st_tr["X"].shape[0],
        "alpha*"    : trained["alpha"],
        "cv_R2"     : trained["cv_score_best"],
        "R2_tr"     : gtr["R2"],   "R2_te"  : gte["R2"],
        "RMSE_tr"   : gtr["RMSE"], "RMSE_te": gte["RMSE"],
        "MAE_te"    : gte["MAE"],
        "bias_te"   : gte["bias"],
        "DW_te"     : gte["DW"],
        "status"    : "ok",
    }


Features esogene : ['SLP', 't2m', 'u10', 'v10', 'wstr_u', 'wstr_v']
Lag set (exo)    : (0, 1, 2, 3, 6, 12, 24, 48, 72, 96, 120, 144, 168)  (13 lag/feat)
AR lags          : ()
p (n. coef)      : 78
L_max (drop)     : 168 h
Window           : −240 / +240 h
CV splits        : 3 (GroupKFold by event)
Peak threshold   : tg_obs_m > 0.8 m, min dist 72 h
Sample weights   : equal_event


## 2. Training multi-evento (split temporale)

Strategia:
- **Split temporale fisso**: i primi `TRAIN_YEARS` anni → training, il resto → test.
  Questo evita ogni leakage temporale tra train e test.
- **Auto-detection** dei picchi con `tg_obs_m > peak_height_m` (default **1 m**).
- **Filtro anti-overlap**: `find_peaks(distance=peak_min_distance_h)` impone una
  distanza minima tra due picchi (default 72 h = 3 giorni). Inoltre, in
  `extract_event_windows`, le finestre di eventi consecutivi vengono **clippate
  a metà strada** così che ogni ora appartenga al massimo a UN evento.
- **Sample weights** $w_t^{(m)} = 1/N_m$ (modalità `equal_event`): ogni evento
  contribuisce uguale alla loss, indipendentemente dalla lunghezza della finestra
  (eventi di bordo o tagliati dall'anti-overlap **non** vengono "premiati" perché
  hanno meno righe).


In [ ]:
# ────────────────────────────────────────────────────────────
#  👉 Split temporale: primi TRAIN_YEARS anni → training, resto → test
# ────────────────────────────────────────────────────────────
TRAIN_YEARS = 5

t_min   = df["time"].min()
t_split = t_min + pd.DateOffset(years=TRAIN_YEARS)
t_max   = df["time"].max()
print(f"TRAIN window : {t_min}  →  {t_split}   ({TRAIN_YEARS} anni)")
print(f"TEST  window : {t_split}  →  {t_max}")

# 1) Auto-detect dei picchi nei due periodi
peaks_train_df = detect_peaks(df, CFG, time_from=t_min,   time_to=t_split)
peaks_test_df  = detect_peaks(df, CFG, time_from=t_split, time_to=t_max + pd.Timedelta(hours=1))
print(f"\nPicchi rilevati > {CFG.peak_height_m} m  (min dist {CFG.peak_min_distance_h} h):")
print(f"  training: {len(peaks_train_df)}     test: {len(peaks_test_df)}")
print("\nPicchi training:")
print(peaks_train_df.to_string(index=False))
print("\nPicchi test:")
print(peaks_test_df.to_string(index=False))

# 2) Estrai finestre evento (con clipping anti-overlap)
train_events = extract_event_windows(df, peaks_train_df["peak"].tolist(), CFG, clip_overlap=True)
print(f"\nEventi di training validi: {len(train_events)} / {len(peaks_train_df)}")
for ev in train_events:
    dur_h = (ev["end"] - ev["start"]).total_seconds() / 3600
    print(f"  · evt {ev['event_id']:>2d}  peak={ev['peak'].strftime('%Y-%m-%d %H:%M')}  "
          f"window={dur_h:5.0f} h  rows={len(ev['df'])}")

# 3) Stack di tutti gli eventi → matrice unica + groups
train_stack = stack_event_matrices(train_events, CFG)
print(f"\nX_train shape : {train_stack['X'].shape}    "
      f"y_train shape: {train_stack['y'].shape}")
print(f"Eventi unici  : {len(np.unique(train_stack['groups']))}    "
      f"p (coef)     : {train_stack['X'].shape[1]}")

# 4) Sample weights (peso uniforme per evento)
peak_heights = {ev["event_id"]: float(peaks_train_df.iloc[ev["event_id"]]["height_m"])
                for ev in train_events}
sw_train = compute_sample_weights(train_stack, CFG, peak_heights=peak_heights)
print(f"\nSample weights mode = '{CFG.weight_mode}'   range=[{sw_train.min():.3g}, {sw_train.max():.3g}]"
      f"  mean={sw_train.mean():.3g}")

# 5) Fit Ridge con GroupKFold-CV pesata
trained = fit_ridge_multi_event(train_stack, CFG, sample_weight=sw_train)
print(f"\n🔧 Ridge α*    = {trained['alpha']:.3g}")
print(f"   #coef       = {len(trained['coef'])}    (intercetta = {trained['intercept']:+.4f})")
print(f"   GroupKFold  = {trained['n_splits']} splits   |   weights = {trained['weight_mode']}")

# 6) Metriche in-sample (globali + per-evento)
df_pe_tr, glob_tr, y_pred_tr, res_tr = evaluate_model(train_stack, trained)
print("\n📊 IN-SAMPLE — globale:")
print(f"   R² = {glob_tr['R2']:+.4f}    RMSE = {glob_tr['RMSE']:.4f} m    "
      f"MAE = {glob_tr['MAE']:.4f} m    bias = {glob_tr['bias']:+.4f} m    "
      f"DW = {glob_tr['DW']:.2f}    n = {glob_tr['n']}")
print("\n📊 IN-SAMPLE — per evento:")
print(df_pe_tr.to_string(index=False,
      formatters={"R2": "{:+.3f}".format, "RMSE": "{:.4f}".format,
                  "MAE": "{:.4f}".format, "bias": "{:+.4f}".format,
                  "DW": "{:.2f}".format}))


## 3. Test su tutti i picchi del periodo successivo (cross-event, out-of-sample)

Predizione su **tutti** i picchi rilevati nella parte di dati dopo lo split (i.e.
gli ultimi anni). Stessa ricetta di lagging, **niente fit** — si applicano solo
`scaler` e `model` salvati in `trained`.


In [ ]:
assert "trained" in globals(), "Esegui prima la cella di TRAINING!"
assert "peaks_test_df" in globals(), "Esegui prima la cella di TRAINING per definire lo split!"

# 1) Estrai finestre dei picchi di test (con clipping anti-overlap)
test_events = extract_event_windows(df, peaks_test_df["peak"].tolist(), CFG, clip_overlap=True)
print(f"Eventi di test validi: {len(test_events)} / {len(peaks_test_df)}")
test_stack = stack_event_matrices(test_events, CFG)
print(f"X_test shape : {test_stack['X'].shape}")

# 2) Predizione (NESSUN re-fit, scaler & model presi da `trained`)
df_pe_te, glob_te, y_pred_te, res_te = evaluate_model(test_stack, trained)

print("\n┏━━━ TEST out-of-sample — GLOBAL ━━━━━━━━━━━━━━━━━━┓")
print(f"  R²    = {glob_te['R2']:+.4f}    (train: {glob_tr['R2']:+.4f})")
print(f"  RMSE  = {glob_te['RMSE']:.4f} m  (train: {glob_tr['RMSE']:.4f} m)")
print(f"  MAE   = {glob_te['MAE']:.4f} m")
print(f"  bias  = {glob_te['bias']:+.4f} m")
print(f"  DW    = {glob_te['DW']:.2f}")
print(f"  n     = {glob_te['n']}")
print("┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛")

print("\n📊 TEST — per evento:")
print(df_pe_te.to_string(index=False,
      formatters={"R2": "{:+.3f}".format, "RMSE": "{:.4f}".format,
                  "MAE": "{:.4f}".format, "bias": "{:+.4f}".format,
                  "DW": "{:.2f}".format}))

# 3) Plot per evento di test: TG, HDM, HDM-corretto + ε vs ε̂
n_te = len(test_events)
fig, axes = plt.subplots(n_te, 2, figsize=(14, 3.0*n_te), squeeze=False)

for row, ev in enumerate(test_events):
    a, b = test_stack["per_event"][ev["event_id"]]
    if b <= a:
        continue
    t_ev   = test_stack["t"][a:b]
    y_ev   = test_stack["y"][a:b]
    yh_ev  = y_pred_te[a:b]

    sub_idx  = ev["df"].set_index("time")
    hdm      = sub_idx.loc[pd.to_datetime(t_ev), "forcoast_p82_m"].to_numpy()
    tg       = sub_idx.loc[pd.to_datetime(t_ev), "tg_obs_m"].to_numpy()
    hdm_corr = hdm - yh_ev
    rmse_raw  = float(np.sqrt(np.mean((hdm      - tg)**2)))
    rmse_corr = float(np.sqrt(np.mean((hdm_corr - tg)**2)))

    ax0 = axes[row, 0]
    ax0.plot(t_ev, tg,       color="#2980B9", lw=1.0, label="TG obs")
    ax0.plot(t_ev, hdm,      color="#E74C3C", lw=1.0, label=f"HDM raw  (RMSE={rmse_raw:.3f})")
    ax0.plot(t_ev, hdm_corr, color="#27AE60", lw=1.1, label=f"HDM corr (RMSE={rmse_corr:.3f})")
    ax0.axvline(ev["peak"], ls=":", color="grey")
    ax0.set_title(f"Event {ev['event_id']} — peak {ev['peak'].strftime('%Y-%m-%d %H:%M')}",
                  fontweight="bold")
    ax0.set_ylabel("Sea level [m]")
    ax0.legend(fontsize=7, loc="upper left")

    ax1 = axes[row, 1]
    ax1.plot(t_ev, y_ev,  color="#8E44AD", lw=0.9, label="ε observed")
    ax1.plot(t_ev, yh_ev, color="#27AE60", lw=1.0, label="ε predicted")
    ax1.axhline(0, color="grey", lw=0.6)
    ax1.axvline(ev["peak"], ls=":", color="grey")
    ax1.set_ylabel("ε [m]")
    ax1.legend(fontsize=7, loc="upper left")

plt.tight_layout()
plt.show()


## 4. Impulse Response Functions e diagnostica residui

I coefficienti $\hat\beta_{i,k}$ sono in unità di "metri per deviazione standard della
feature". Se il modello cattura un meccanismo fisico reale, le IRF dovrebbero essere
**stabili** (cambiando il set di eventi di training non dovrebbero stravolgersi).


In [ ]:
# 4.1 IRF
beta = plot_irfs(trained, title_suffix=f"— {STATION_NAME}")

# 4.2 Diagnostica residui — TRAINING
plot_residual_diagnostics(res_tr, t=train_stack["t"],
                          title="Diagnostica residui — TRAIN (in-sample)")

# 4.3 Diagnostica residui — TEST
plot_residual_diagnostics(res_te, t=test_stack["t"],
                          title="Diagnostica residui — TEST (out-of-sample)")


## 5. Significato statistico del setup *grouped multi-event Ridge*

**Cosa abbiamo fatto, in sintesi.**
1. Abbiamo costruito $M$ piccoli problemi di regressione (uno per evento) e li abbiamo
   **concatenati** in un'unica $X \in \mathbb{R}^{N\times p}$ con $N = \sum_m N_m$,
   $p = n_f(L+1) = 292$.
2. Il vettore `groups` ricorda **da quale evento** proviene ogni riga.
3. La standardizzazione è fatta **una volta sola** sull'intera $X$ di training
   (no per-evento → la stessa σ definisce l'unità di misura di tutte le IRF).
4. Ridge minimizza $\|\mathbf{y}-Z\boldsymbol\beta\|^2 + \alpha\|\boldsymbol\beta\|^2$.
   Il termine $\alpha\|\boldsymbol\beta\|^2$ è essenziale perché $p\approx 292$ è
   confrontabile con $N_m \approx 400$ per singolo evento; senza Ridge, ogni evento
   "ruberebbe" gradi di libertà al successivo e i coefficienti diventerebbero instabili.
5. **GroupKFold** assicura che durante la CV un intero evento (tutte le sue righe
   correlate temporalmente) finisca o nel train o nel validation di un fold, **mai
   in entrambi**. Questo è cruciale perché:
   - righe consecutive entro lo stesso evento sono **fortemente autocorrelate**
     (lag 0..72, finestra di 20 giorni → l'autocorrelazione decade molto lentamente);
   - un k-fold "vanilla" mescolerebbe pezzi dello stesso evento tra fold, gonfiando
     artificialmente il punteggio CV e portando a $\alpha$ troppo basso (overfit).

**Interpretazione delle IRF $\hat\beta_{i,k}$.**
Sono la "funzione di trasferimento empirica" tra ciascuna forzante atmosferica e
l'errore residuo della HDM. Per esempio, una IRF di $u_{10}$ con valori positivi
intorno al lag 10–24 h indica che vento da ovest sostenuto un giorno prima del
picco genera errore positivo (HDM > TG ⇒ il modello sovrastima il livello).

**Cosa ci dice il confronto train/test.**
- Se $R^2_\text{te}\approx R^2_\text{tr}$ e le metriche per-evento sono omogenee →
  la dinamica è **stazionaria** tra tempeste: il modello cattura un meccanismo fisico.
- Se $R^2_\text{te}$ varia molto da evento a evento → le tempeste hanno regimi diversi
  (per es. direzione di propagazione, surge remoto, seiches) che 4 forzanti locali
  non possono distinguere.
- Bias sistematico per-evento → c'è una componente quasi-statica/stagionale non
  catturata (tide gauge datum, atmosferica media, steric).

**Limiti consapevoli.**
- Modello solo locale: nessuna forzante che descriva la circolazione del Mare del Nord
  o la propagazione baroclina dal Mar Baltico.
- Stesse $L$ per tutte le feature: SLP risponde più velocemente, vento più lentamente
  → si potrebbe pensare a $L_i$ feature-specifici.
- Ridge fa shrinkage uniforme: con un *Lasso* o *Group-Lasso* potremmo ottenere IRF
  più sparse e interpretabili (un coefficiente per lag attivo).

---

## 6. Estensione futura: predittori satellitari offshore

Per integrare osservazioni satellitari (altimetria SAR/SARAL/Sentinel-3/6, SWOT) come
predittori esogeni aggiuntivi, la pipeline non cambia struttura — basta:

1. **Ricampionare** la traccia satellitare alla griglia oraria della stazione tramite
   interpolazione spazio-temporale al *control point* offshore più vicino (o a una
   media pesata su un riquadro). Output: serie oraria $\eta_\text{sat}(t)$ per ogni
   variabile satellitare (sea-level anomaly, SWH, sigma0…).
2. **Aggiungere le nuove colonne** al DataFrame `df` con gli stessi nomi nelle
   `Config.features = ("SLP", "t2m", "u10", "v10", "ssha_sat", "swh_sat")`.
3. **Gestire i gap**: i satelliti hanno revisit di 10–35 giorni → riempire con
   media climatologica o usare un *missingness flag* come feature aggiuntiva
   (`x_missing(t) ∈ {0,1}`); in alternativa modellare un'unica variabile satellitare
   "smoothed" (es. Kalman/Gauss-Markov).
4. **Lag dedicati**: il segnale altimetrico riflette anomalie offshore che si
   propagano verso costa con tempi lunghi → si potrebbe alzare $L$ a 96–120 h
   *solo* per le feature satellitari, mantenendo $L=72$ per le forzanti atmosferiche
   (lag-horizon feature-specifico).
5. **Group-Lasso** sui blocchi di colonne `feature × lag` per selezionare quali
   predittori (atmosferici vs satellitari) entrano nel modello → utile per scoprire
   se l'aggiunta del satellite è statisticamente giustificata.
6. **CV stratificata per stagione**: oltre a GroupKFold per evento, si potrebbe
   bilanciare i fold rispetto al periodo dell'anno (la copertura satellitare e i
   regimi di mareggiata variano fra estate/inverno).
7. **Diagnostica di significatività**: confronto $R^2$ con e senza satellite, test
   F approssimato, oppure (più robusto) bootstrap sui residui *per evento*.

In quel momento il modello passerà da puro "post-processore locale dell'errore HDM"
a un vero modello di **data-fusion** tra forzanti meteo locali e osservazioni
satellitari di livello del mare offshore.


## 7. Esperimento comparativo — griglia di configurazioni (København)

Invece di un singolo modello "finale" mettiamo il notebook in modalità
**comparative experiment**: facciamo girare la stessa pipeline al variare di
diversi iperparametri/feature-set e raccogliamo tutte le metriche in **un'unica
tabella**, così che le scelte modellistiche siano giustificate da numeri e non
da intuizione.

Assi che variamo qui:

1. **Lag massimo $L_\max$** (sotto-campionamento dei lag): cattura quanto
   indietro nel tempo guarda il modello.
2. **HDM come feature esogena** (`include_hdm`): il modello può usare lo
   stato corrente del proprio HDM come predittore della propria correzione.
3. **AR-lags del target** $\varepsilon_{t-1},\dots$ (`ar_lags`): aggiunge
   memoria del residuo passato → il modello diventa **ARX**, quindi può
   sfruttare l'autocorrelazione che vediamo nelle ACF dei residui senza dover
   inserire una lista infinita di lag esogeni.

CV: **GroupKFold a 3 fold** (più robusto con ~10–15 eventi di training).
Tutte le configurazioni usano lo stesso split temporale e gli stessi pesi
$w_t = 1/N_m$ per evento, quindi le metriche sono direttamente confrontabili.


In [3]:
# ── 7.1 Esperimento comparativo FOCAL: 6 configurazioni ARX (L_exog = 24 h)
# Specifiche richieste dall'utente:
# - L_exog fisso = (0,1,2,3,6,12,24)
# - AR sets = (1,2,3,6,12), (1,2,3,6,12,24), (1,2,3,6,12,24,48)
# - HDM toggle = False / True (se True: includi forcoast_p82_m tra le feature)
# - Feature set: SLP, t2m, u10, v10, tidal_12h, tidal_24h, wstr_u, wstr_v

L_EXOG = (0, 1, 2, 3, 6, 12, 24)
AR_SETS = [
    (1, 2, 3, 6, 12),
    (1, 2, 3, 6, 12, 24),
    (1, 2, 3, 6, 12, 24, 48),
]
HDM_OPTIONS = [False, True]

# Usa la variabile TRAIN_YEARS definita sopra (default nel notebook = 5).
TRAIN_YEARS_EXP = TRAIN_YEARS

FEATURES_FOCAL = ("SLP", "t2m", "u10", "v10", "tidal_12h", "tidal_24h", "wstr_u", "wstr_v")

results = []
config_records = []
for ar in AR_SETS:
    for hdm in HDM_OPTIONS:
        label = f"L=24h | HDM={'Y' if hdm else 'N'} | AR{ar}"
        kw = dict(
            features=FEATURES_FOCAL,
            lags=L_EXOG,
            add_wind_stress=False,     # wstr_* sono già nelle FEATURES_FOCAL
            include_hdm=hdm,
            ar_lags=ar,
            cv_n_splits=3,
            pre_event_hours=240,
            post_event_hours=240,
            peak_height_m=CFG.peak_height_m,
            peak_min_distance_h=CFG.peak_min_distance_h,
            weight_mode=CFG.weight_mode,
        )
        try:
            row = run_one_experiment(df, kw, train_years=TRAIN_YEARS_EXP, label=label)
        except Exception as e:
            row = {"label": label, "status": f"ERR: {type(e).__name__}: {e}"}
        results.append(row)
        config_records.append((label, kw))
        print(f"  ✓ {label:48s}  →  R²_te = {row.get('R2_te', float('nan')):+.3f}   α* = {row.get('alpha*', float('nan'))}")

# Tabella comparativa finale
res_df = pd.DataFrame(results)
cols_order = ["label", "n_train_ev", "n_test_ev", "p_coef", "N_train", "p_over_N",
              "alpha*", "cv_R2", "R2_tr", "R2_te", "RMSE_tr", "RMSE_te",
              "MAE_te", "bias_te", "DW_te", "status"]
cols_order = [c for c in cols_order if c in res_df.columns]
res_df = res_df[cols_order]
res_df_sorted = res_df.sort_values("R2_te", ascending=False, na_position="last").reset_index(drop=True)

print("\n" + "═"*120)
print(f"  RISULTATI FOCAL — stazione: {STATION_NAME}   (train={TRAIN_YEARS_EXP}y, GroupKFold-3)")
print("═"*120)
print(res_df_sorted.to_string(
    index=False,
    formatters={
        "p_over_N": "{:.3f}".format,
        "alpha*"  : "{:.2g}".format,
        "cv_R2"   : "{:+.3f}".format,
        "R2_tr"   : "{:+.3f}".format,
        "R2_te"   : "{:+.3f}".format,
        "RMSE_tr" : "{:.4f}".format,
        "RMSE_te" : "{:.4f}".format,
        "MAE_te"  : "{:.4f}".format,
        "bias_te" : "{:+.4f}".format,
        "DW_te"   : "{:.2f}".format,
    },
))
print("═"*120)
print(f"\nN. configurazioni testate: {len(res_df)}   (3 AR-sets × 2 HDM = 6)")
if len(res_df_sorted):
    print(f"Migliore (per R²_te)  : {res_df_sorted.iloc[0]['label']}   →   R²_te = {res_df_sorted.iloc[0]['R2_te']:+.4f}")

# ── Rerun per la miglior configurazione: ottieni trained, stacks e diagnostica
if len(res_df_sorted) == 0:
    print("Nessuna configurazione valida → interrompo diagnostica.")
else:
    best_label = res_df_sorted.iloc[0]['label']
    idx_best = [i for i, (lab, kw) in enumerate(config_records) if lab == best_label][0]
    best_kw = config_records[idx_best][1]

    print(f"\nRi-eseguo la miglior configurazione per diagnosticare: {best_label}")
    cfg = Config(**best_kw)
    t_min = df['time'].min(); t_split = t_min + pd.DateOffset(years=TRAIN_YEARS_EXP); t_max = df['time'].max()

    pk_tr = detect_peaks(df, cfg, t_min, t_split)
    pk_te = detect_peaks(df, cfg, t_split, t_max + pd.Timedelta(hours=1))
    ev_tr = extract_event_windows(df, pk_tr['peak'].tolist(), cfg, clip_overlap=True)
    ev_te = extract_event_windows(df, pk_te['peak'].tolist(), cfg, clip_overlap=True)

    st_tr = stack_event_matrices(ev_tr, cfg)
    st_te = stack_event_matrices(ev_te, cfg)
    ph = {e['event_id']: float(pk_tr.iloc[e['event_id']]['height_m']) for e in ev_tr}
    sw_tr = compute_sample_weights(st_tr, cfg, peak_heights=ph)

    trained_best = fit_ridge_multi_event(st_tr, cfg, sample_weight=sw_tr)
    df_pe_tr, glob_tr, y_pred_tr, res_tr = evaluate_model(st_tr, trained_best)
    df_pe_te, glob_te, y_pred_te, res_te = evaluate_model(st_te, trained_best)

    print('\n📌 Metriche miglior modello — TRAIN')
    print(f"  R2 = {glob_tr['R2']:+.4f}   RMSE = {glob_tr['RMSE']:.4f} m   n = {glob_tr['n']}")
    print('\n📌 Metriche miglior modello — TEST')
    print(f"  R2 = {glob_te['R2']:+.4f}   RMSE = {glob_te['RMSE']:.4f} m   n = {glob_te['n']}")

    # IRF per il miglior modello
    beta = plot_irfs(trained_best, title_suffix=f"— BEST: {best_label} — {STATION_NAME}")

    # Diagnostica residui (globale)
    plot_residual_diagnostics(res_tr, t=st_tr['t'], title=f"Residui TRAIN — {best_label}")
    plot_residual_diagnostics(res_te, t=st_te['t'], title=f"Residui TEST — {best_label}")

    # Plot per-event di TEST (HDM raw vs HDM corrected + ε vs ε̂)
    n_te = len(ev_te)
    if n_te > 0:
        fig, axes = plt.subplots(n_te, 2, figsize=(14, 3.0*n_te), squeeze=False)
        for row, ev in enumerate(ev_te):
            a, b = st_te['per_event'][ev['event_id']]
            if b <= a: continue
            t_ev = st_te['t'][a:b]; y_ev = st_te['y'][a:b]; yh_ev = y_pred_te[a:b]
            sub_idx = ev['df'].set_index('time')
            if 'forcoast_p82_m' in ev['df'].columns:
                hdm = sub_idx.loc[pd.to_datetime(t_ev), 'forcoast_p82_m'].to_numpy()
            else:
                hdm = np.full_like(yh_ev, np.nan)
            tg = sub_idx.loc[pd.to_datetime(t_ev), 'tg_obs_m'].to_numpy()
            hdm_corr = hdm - yh_ev
            rmse_raw = float(np.sqrt(np.mean((hdm - tg)**2)))
            rmse_corr = float(np.sqrt(np.mean((hdm_corr - tg)**2)))

            ax0 = axes[row, 0]
            ax0.plot(t_ev, tg, color="#2980B9", lw=1.0, label="TG obs")
            ax0.plot(t_ev, hdm, color="#E74C3C", lw=1.0, label=f"HDM raw  (RMSE={rmse_raw:.3f})")
            ax0.plot(t_ev, hdm_corr, color="#27AE60", lw=1.1, label=f"HDM corr (RMSE={rmse_corr:.3f})")
            ax0.axvline(ev['peak'], ls=":", color="grey")
            ax0.set_title(f"Event {ev['event_id']} — peak {ev['peak'].strftime('%Y-%m-%d %H:%M')}")
            ax0.set_ylabel('Sea level [m]')
            ax0.legend(fontsize=7, loc='upper left')

            ax1 = axes[row, 1]
            ax1.plot(t_ev, y_ev, color="#8E44AD", lw=0.9, label="ε observed")
            ax1.plot(t_ev, yh_ev, color="#27AE60", lw=1.0, label="ε predicted")
            ax1.axhline(0, color="grey", lw=0.6)
            ax1.axvline(ev['peak'], ls=":", color="grey")
            ax1.set_ylabel('ε [m]')
            ax1.legend(fontsize=7, loc='upper left')
        plt.tight_layout(); plt.show()

    # Tabella per-evento del test
    if isinstance(df_pe_te, pd.DataFrame) and not df_pe_te.empty:
        print('\n📋 Tabella metriche per-evento (TEST):')
        print(df_pe_te.to_string(index=False,
              formatters={"R2": "{:+.3f}".format, "RMSE": "{:.4f}".format,
                          "MAE": "{:.4f}".format, "bias": "{:+.4f}".format,
                          "DW": "{:.2f}".format}))

    # Salva l'oggetto trained_best nel namespace per eventuali usi successivi
    trained = trained_best
    train_stack = st_tr
    test_stack = st_te
    print('\n✅ Diagnostic run completa. Oggetto `trained` disponibile nel workspace.')


NameError: name 'TRAIN_YEARS' is not defined

## 8. Confronto onesto: NO_AR vs PURE_PERSISTENCE vs FULL_ARX

Tre famiglie di modelli a confronto, **stesso pipeline, stessi eventi, stesso split, stessa CV**:

| Modello             | Esogene fisiche | AR-lag di ε |
| ------------------- | --------------- | ----------- |
| **NO_AR**           | ✅ (SLP, t2m, u10, v10, tidal_12h, tidal_24h, wstr_u, wstr_v) — `L_exog = 24 h` | ❌ |
| **PURE_PERSISTENCE**| ❌                                                                              | ✅ `[1,2,3,6,12,24]` |
| **FULL_ARX**        | ✅                                                                              | ✅ |

Obiettivo:
- isolare quanto è spiegato dalle **forzanti fisiche** (NO_AR);
- quanto è spiegato dalla **persistenza del residuo** (PURE_PERSISTENCE);
- quanto valore **incrementale** porta combinarle (FULL_ARX).

HDM **non** è incluso tra le feature in questa comparazione.


In [5]:
# ── 8.1 Fit & evaluate dei tre modelli (NO_AR, PURE_PERSISTENCE, FULL_ARX)
# Stesso split temporale, stessi eventi, stessa GroupKFold, stesso scaler logic.

EXO_FEATURES_CMP = ("SLP", "t2m", "u10", "v10", "tidal_12h", "tidal_24h", "wstr_u", "wstr_v")
L_EXOG_CMP       = (0, 1, 2, 3, 6, 12, 24)         # 24 h horizon
AR_LAGS_CMP      = (1, 2, 3, 6, 12, 24)
TRAIN_YEARS_CMP  = 5                      # riusa lo split temporale del notebook

MODEL_SPECS = {
    "NO_AR":            dict(features=EXO_FEATURES_CMP, lags=L_EXOG_CMP, ar_lags=()),
    "PURE_PERSISTENCE": dict(features=(),               lags=(),         ar_lags=AR_LAGS_CMP),
    "FULL_ARX":         dict(features=EXO_FEATURES_CMP, lags=L_EXOG_CMP, ar_lags=AR_LAGS_CMP),
}

def _fit_and_eval_one(df, name, spec, train_years):
    """Crea un Config ad-hoc, ri-usa l'intero pipeline esistente e ritorna trained+stacks+metrics."""
    # NB: per PURE_PERSISTENCE features=() e add_wind_stress=False → all_features=[].
    cfg = Config(
        features=spec["features"],
        lags=spec["lags"] if len(spec["lags"]) else (0,),  # placeholder per non rompere max(); non si usa se features=()
        ar_lags=spec["ar_lags"],
        add_wind_stress=False,
        include_hdm=False,
        cv_n_splits=3,
        pre_event_hours=240,
        post_event_hours=240,
        peak_height_m=CFG.peak_height_m,
        peak_min_distance_h=CFG.peak_min_distance_h,
        weight_mode=CFG.weight_mode,
    )

    # Workaround pulito: se non ci sono feature esogene, by-passiamo build_lag_matrix usando
    # un wrapper che costruisce solo la parte AR. Implementato qui inline per non toccare il pipeline.
    t_min = df["time"].min(); t_split = t_min + pd.DateOffset(years=train_years); t_max = df["time"].max()
    pk_tr = detect_peaks(df, cfg, t_min, t_split)
    pk_te = detect_peaks(df, cfg, t_split, t_max + pd.Timedelta(hours=1))
    ev_tr = extract_event_windows(df, pk_tr["peak"].tolist(), cfg, clip_overlap=True)
    ev_te = extract_event_windows(df, pk_te["peak"].tolist(), cfg, clip_overlap=True)

    def _build_pure_ar(sub, ar_lags, target):
        L = max(ar_lags)
        y_full = sub[target].to_numpy()
        cols, names = [], []
        for k in ar_lags:
            shifted = np.empty_like(y_full, dtype=float); shifted[:k] = np.nan
            shifted[k:] = y_full[:-k]
            cols.append(shifted); names.append(f"{target}_arlag{k}")
        X = np.column_stack(cols)[L:]
        y = y_full[L:]
        t = sub["time"].to_numpy()[L:]
        return X, y, t, names

    def _stack(events):
        Xs, ys, ts, gs = [], [], [], []
        per_event, cursor, feat_names = {}, 0, None
        for ev in events:
            if len(cfg.all_features) == 0:
                Xe, ye, te, names = _build_pure_ar(ev["df"], cfg.ar_lags, cfg.target)
            else:
                Xe, ye, te, names = build_lag_matrix(ev["df"], cfg.all_features, cfg.lags,
                                                    cfg.target, cfg.ar_lags)
            if Xe.shape[0] == 0:
                per_event[ev["event_id"]] = (cursor, cursor); continue
            Xs.append(Xe); ys.append(ye); ts.append(te)
            gs.append(np.full(Xe.shape[0], ev["event_id"], dtype=int))
            per_event[ev["event_id"]] = (cursor, cursor + Xe.shape[0])
            cursor += Xe.shape[0]; feat_names = names
        return {"X": np.vstack(Xs), "y": np.concatenate(ys),
                "t": np.concatenate(ts), "groups": np.concatenate(gs),
                "per_event": per_event, "feat_names": feat_names}

    st_tr = _stack(ev_tr); st_te = _stack(ev_te)
    ph    = {e["event_id"]: float(pk_tr.iloc[e["event_id"]]["height_m"]) for e in ev_tr}
    sw_tr = compute_sample_weights(st_tr, cfg, peak_heights=ph)
    trained = fit_ridge_multi_event(st_tr, cfg, sample_weight=sw_tr)

    df_pe_tr, glob_tr, y_pred_tr, res_tr = evaluate_model(st_tr, trained)
    df_pe_te, glob_te, y_pred_te, res_te = evaluate_model(st_te, trained)

    row = {
        "label"     : name,
        "n_train_ev": len(ev_tr), "n_test_ev": len(ev_te),
        "p_coef"    : st_tr["X"].shape[1], "N_train": st_tr["X"].shape[0],
        "alpha*"    : trained["alpha"],
        "cv_R2"     : trained["cv_score_best"],
        "R2_tr"     : glob_tr["R2"],   "R2_te"  : glob_te["R2"],
        "RMSE_tr"   : glob_tr["RMSE"], "RMSE_te": glob_te["RMSE"],
        "MAE_te"    : glob_te["MAE"],
        "bias_te"   : glob_te["bias"],
        "DW_te"     : glob_te["DW"],
    }
    return {
        "row": row, "trained": trained, "cfg": cfg,
        "ev_tr": ev_tr, "ev_te": ev_te,
        "st_tr": st_tr, "st_te": st_te,
        "y_pred_tr": y_pred_tr, "y_pred_te": y_pred_te,
        "df_pe_tr": df_pe_tr, "df_pe_te": df_pe_te,
    }

cmp_results = {}
for name, spec in MODEL_SPECS.items():
    print(f"▶ Fitting {name} ...", flush=True)
    cmp_results[name] = _fit_and_eval_one(df, name, spec, TRAIN_YEARS_CMP)
    r = cmp_results[name]["row"]
    print(f"   R²_te={r['R2_te']:+.4f}   RMSE_te={r['RMSE_te']:.4f}   p={r['p_coef']}   α*={r['alpha*']:.2g}")

# Tabella comparativa
cmp_df = pd.DataFrame([cmp_results[n]["row"] for n in MODEL_SPECS])
cmp_df = cmp_df[["label","n_train_ev","n_test_ev","p_coef","N_train",
                 "alpha*","cv_R2","R2_tr","R2_te","RMSE_tr","RMSE_te",
                 "MAE_te","bias_te","DW_te"]]
print("\n" + "═"*110)
print(f"  CONFRONTO 3 MODELLI — {STATION_NAME}   (train={TRAIN_YEARS_CMP}y, GroupKFold-3)")
print("═"*110)
print(cmp_df.to_string(index=False, formatters={
    "alpha*":"{:.2g}".format, "cv_R2":"{:+.3f}".format,
    "R2_tr":"{:+.3f}".format, "R2_te":"{:+.3f}".format,
    "RMSE_tr":"{:.4f}".format, "RMSE_te":"{:.4f}".format,
    "MAE_te":"{:.4f}".format, "bias_te":"{:+.4f}".format, "DW_te":"{:.2f}".format,
}))
print("═"*110)

# Valore incrementale FULL_ARX vs PURE_PERSISTENCE
r_arx = cmp_results["FULL_ARX"]["row"]; r_per = cmp_results["PURE_PERSISTENCE"]["row"]
r_exo = cmp_results["NO_AR"]["row"]
delta_R2_test   = r_arx["R2_te"]   - r_per["R2_te"]
delta_RMSE_test = r_per["RMSE_te"] - r_arx["RMSE_te"]   # >0 ⇒ ARX migliora (RMSE più basso)
print("\n📐 Valore incrementale di FULL_ARX rispetto a PURE_PERSISTENCE:")
print(f"   ΔR²(test)   = R²(ARX) − R²(PERS)     = {r_arx['R2_te']:+.4f} − {r_per['R2_te']:+.4f} = {delta_R2_test:+.4f}")
print(f"   ΔRMSE(test) = RMSE(PERS) − RMSE(ARX) = {r_per['RMSE_te']:.4f} − {r_arx['RMSE_te']:.4f} = {delta_RMSE_test:+.4f} m  (positivo = miglioramento)")
print("\n📐 Valore incrementale di FULL_ARX rispetto a NO_AR (forzanti sole):")
print(f"   ΔR²(test)   = {r_arx['R2_te'] - r_exo['R2_te']:+.4f}")
print(f"   ΔRMSE(test) = {r_exo['RMSE_te'] - r_arx['RMSE_te']:+.4f} m")


▶ Fitting NO_AR ...
   R²_te=-1.5951   RMSE_te=0.1268   p=56   α*=1e+03
▶ Fitting PURE_PERSISTENCE ...
   R²_te=+0.8956   RMSE_te=0.0254   p=6   α*=0.001
▶ Fitting FULL_ARX ...
   R²_te=+0.9009   RMSE_te=0.0248   p=62   α*=1

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  CONFRONTO 3 MODELLI — København   (train=5y, GroupKFold-3)
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
           label  n_train_ev  n_test_ev  p_coef  N_train alpha*  cv_R2  R2_tr  R2_te RMSE_tr RMSE_te MAE_te bias_te DW_te
           NO_AR          15          9      56     5349  1e+03 +0.113 +0.342 -1.595  0.1147  0.1268 0.1040 +0.0863  0.05
PURE_PERSISTENCE          15          9       6     5349  0.001 +0.958 +0.964 +0.896  0.0268  0.0254 0.0189 +0.0008  2.10
        FULL_ARX          15          9      62     5349      1 +0.960 +0.967 +0.901  0.0255  0.0248 0.0184 +0.0015  2.08
═══